# Multiprocess

## Forking

In [1]:
import os 
pid_list = [] 
def main(): 
    pid_list.append(os.getpid()) 
    child_pid = os.fork() 
 
    if child_pid == 0: 
        pid_list.append(os.getpid()) 
        print() 
        print("CHLD: hey, I am the child process") 
        print("CHLD: all the pids I know %s" % pid_list) 
 
    else: 
        pid_list.append(os.getpid()) 
        print() 
        print("PRNT: hey, I am the parent process") 
        print("PRNT: the child pid is %d" % child_pid) 
        print("PRNT: all the pids I know %s" % pid_list) 
 
 
if __name__ == "__main__": 
    main()


PRNT: hey, I am the parent process
PRNT: the child pid is 304106
PRNT: all the pids I know [304019, 304019]

CHLD: hey, I am the child process
CHLD: all the pids I know [304019, 304106]


## Multiprocessing

In [4]:
from multiprocessing import Process
import os
def work(identifier):
    print(
        f'Hey, I am the process '
        f'{identifier}, pid: {os.getpid()}'
    )
def main():
    processes = [
        Process(target=work, args=(number,))
        for number in range(5)
    ]
    for process in processes:
        process.start()
    while processes:
        processes.pop().join()
if __name__ == "__main__":
    main()

Hey, I am the process 2, pid: 304424
Hey, I am the process 0, pid: 304420Hey, I am the process 3, pid: 304427
Hey, I am the process 4, pid: 304430

Hey, I am the process 1, pid: 304422


## Multiprocessing with Pipe interprocess communication

In [5]:
def worker(connection):
    while True:
        instance = connection.recv()
        if instance:
            print(f"CHLD: recv: {instance}")
        if instance is None:
            break

In [6]:
from multiprocessing import Process, Pipe 
 
class CustomClass: 
    pass 
 
def main():
    parent_conn, child_conn = Pipe()
    child = Process(target=worker, args=(child_conn,))
    for item in (
        42,
        'some string',
        {'one': 1},
        CustomClass(),
        None,
    ):
        print(
            "PRNT: send: {}".format(item)
        )
        parent_conn.send(item)
    child.start()
    child.join() 
if __name__ == "__main__": 
    main()

PRNT: send: 42
PRNT: send: some string
PRNT: send: {'one': 1}
PRNT: send: <__main__.CustomClass object at 0x7f3630361270>
PRNT: send: None
CHLD: recv: 42
CHLD: recv: some string
CHLD: recv: {'one': 1}
CHLD: recv: <__main__.CustomClass object at 0x7f3630362140>
